# 12 — Native missing-value handling instead of median imputation

**Goal:** the current production pipeline (`src/model.py::make_baseline_model`) runs
every feature through `SimpleImputer(strategy="median")` before
`HistGradientBoostingRegressor`. Several feature columns can genuinely be missing for
reasons that carry information, not just noise: `tws_neighbour_mean`/
`tws_local_deviation` (NaN when no neighbour cell has an observed value that month),
`tws_climatology_mean`/`tws_climatology_deviation` (NaN for a cell's first occurrence
of a calendar month, i.e. no earlier year to average), and `spei12_prior_value`/
`spei12_prior_age` (NaN when a cell has no strictly-earlier SPEI_12 reading at all).
Median-imputing these throws away the "this was missing" signal for all of them -
`TWS_t` itself is the only one this project already compensates for directly, via
`months_since_anchor`.

**Source**: found by reviewing Kaggle competition/technique write-ups this session
("Gradient Boosting Explained - Ensemble Learning" noted XGBoost's default-direction
split for missing values) and confirmed directly in this project's own installed
scikit-learn 1.7.2 source
(`sklearn/ensemble/_hist_gradient_boosting/gradient_boosting.py`, class docstring,
lines 1477-1483): `HistGradientBoostingRegressor` has native missing-value support -
"the tree grower learns at each split point whether samples with missing values should
go to the left or right child, based on the potential gain" - the same underlying idea
LightGBM/XGBoost use (this implementation is explicitly inspired by LightGBM, per the
same docstring). This is a documented, first-class capability, not a workaround.

**Test**: single controlled change - drop `SimpleImputer` entirely and feed
`HistGradientBoostingRegressor` the raw (NaN-containing) feature matrix directly - on
the existing 5-seed `mask_augmented_horizon_matched_split` proxy, current default
hyperparameters otherwise unchanged.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import os

import threadpoolctl

N_THREADS = max(1, os.cpu_count() // 2)  # edit to taste; os.cpu_count() = logical CPUs
_thread_limiter = threadpoolctl.threadpool_limits(limits=N_THREADS)
print(f"Capped native thread pools (OpenMP/BLAS) at {N_THREADS} of {os.cpu_count()} logical CPUs.")

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from src import config, data, evaluate, features, model
from src.train import get_feature_cols

pd.set_option("display.width", 120)


Capped native thread pools (OpenMP/BLAS) at 11 of 22 logical CPUs.


## Load data (fast)

In [2]:
raw_train, test, _ = data.load_raw_data()
target_horizons = evaluate.compute_test_horizons(raw_train, test)
masked_month_fraction, masked_row_fraction = evaluate.measure_masking_pattern(test)
print(f"target_horizons: {sorted(target_horizons)}")
print(f"masked_month_fraction={masked_month_fraction:.4f} masked_row_fraction={masked_row_fraction:.4f}")


target_horizons: [1, 5, 6, 7, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 35, 39, 40]
masked_month_fraction=0.6667 masked_row_fraction=0.9978


## How much missingness are we actually imputing away?

Quick check before the model comparison - measure the real NaN rate per feature
column on one fit split, so the test below isn't run blind.


In [3]:
fit_df, val_df = evaluate.mask_augmented_horizon_matched_split(
    raw_train, target_horizons, masked_month_fraction, masked_row_fraction,
    fit_seed=0, val_seed=100,
)
feature_cols = get_feature_cols(fit_df)
X_fit_check = features.select_base_features(fit_df, feature_cols)
nan_rates = pd.Series(np.isnan(X_fit_check).mean(axis=0), index=feature_cols).sort_values(ascending=False)
print(nan_rates)


tws_climatology_deviation    0.299503
tws_climatology_mean         0.190278
tws_local_deviation          0.081538
months_since_anchor          0.081086
TWS_t                        0.081086
tws_neighbour_mean           0.039157
spei12_prior_value           0.009153
spei12_prior_age             0.009153
SPEI_06_t                    0.000000
SPEI_12_t                    0.000000
SPEI_01_t                    0.000000
month_cos                    0.000000
month_sin                    0.000000
SPEI_03_t                    0.000000
SOIL_MOISTURE_t              0.000000
dtype: float64


## 5-seed comparison: `SimpleImputer(median)` vs. native NaN handling

Same hyperparameters as `make_baseline_model()`, same 5-seed
`mask_augmented_horizon_matched_split` proxy this project already uses to gate every
feature/hyperparameter change - only the missing-value handling differs.


In [4]:
def make_model_native_nan():
    return HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=300,
        max_depth=8,
        min_samples_leaf=50,
        l2_regularization=1.0,
        random_state=config.RANDOM_STATE,
    )


def five_seed_mean_rmse(make_model, uses_pipeline):
    rmses = []
    for seed in range(5):
        fit_s, val_s = evaluate.mask_augmented_horizon_matched_split(
            raw_train, target_horizons, masked_month_fraction, masked_row_fraction,
            fit_seed=seed, val_seed=seed + 100,
        )
        cols = get_feature_cols(fit_s)
        Xf = features.select_base_features(fit_s, cols)
        yf = fit_s[config.TARGET_COL].to_numpy()
        Xv = features.select_base_features(val_s, cols)
        yv = val_s[config.TARGET_COL].to_numpy()

        m = make_model()
        m.fit(Xf, yf)
        pred = model.predict(m, Xv) if uses_pipeline else m.predict(Xv)
        rmses.append(evaluate.rmse(yv, pred))
    return rmses


imputed_rmses = five_seed_mean_rmse(model.make_baseline_model, uses_pipeline=True)
native_nan_rmses = five_seed_mean_rmse(make_model_native_nan, uses_pipeline=False)

print(f"SimpleImputer(median) [current production]: mean={np.mean(imputed_rmses):.4f} "
      f"range=({min(imputed_rmses):.4f}, {max(imputed_rmses):.4f})  "
      f"rmses={[round(r, 4) for r in imputed_rmses]}")
print(f"Native NaN handling (no imputer):           mean={np.mean(native_nan_rmses):.4f} "
      f"range=({min(native_nan_rmses):.4f}, {max(native_nan_rmses):.4f})  "
      f"rmses={[round(r, 4) for r in native_nan_rmses]}")


SimpleImputer(median) [current production]: mean=0.7117 range=(0.6925, 0.7276)  rmses=[0.7276, 0.7223, 0.716, 0.6925, 0.7002]
Native NaN handling (no imputer):           mean=0.7101 range=(0.6926, 0.7235)  rmses=[0.7235, 0.7212, 0.7113, 0.6926, 0.7021]


## Gate decision

**Proxy result**: native NaN handling wins 3/5 seeds clearly (seeds 0/1/2: -0.0041,
-0.0011, -0.0047) and is within noise on the other 2 (seed 3: +0.0001, seed 4:
+0.0019) - mean 0.7117 -> 0.7101 (~0.23%). A small-but-mostly-consistent positive
signal, the same shape as P2's SPEI_12 prior-reading candidate that was confirmed
real by a submission after initially looking "too weak" on proxy alone (see
`RESOURCES.md`, `feedback_confirm_weak_proxy_signals_with_real_submission`) - so per
that established policy, this is exactly the kind of signal that should be checked
for real, not written off.

This is a pipeline-level change (touches how every feature's missingness is handled,
not a single new feature), so per this project's practice since P3, it needs a real
Zindi submission before graduating regardless of the proxy direction - a proxy win on
a pipeline/hyperparameter-style change has inverted on the real leaderboard before.

**Submission generated**: `outputs/submission_native_nan.csv` (single controlled
change vs. the current production pipeline - `SimpleImputer` removed,
`HistGradientBoostingRegressor` fed the raw NaN-containing feature matrix directly,
everything else identical to `src/train.py`'s final fit). Validated: 280,961 rows
(matches `SampleSubmission.csv`), `ID` order matches exactly, no NaN predictions,
prediction range (-2.42, 2.48) consistent with prior submissions' scale. Queued for
upload - fill in the real score here once back, and update `README.md`/`RESOURCES.md`
either way (graduate `src/model.py` on a real win, or log as a second pipeline-level
negative result if it inverts like P3 did).
